> 在线讲义：[第 31 章 · 微分几何与流形优化：切空间、收缩映射、测地线、黎曼梯度](https://wikieden.github.io/math-and-eval-stats/lessons/31-manifold-optimization.html) · 本书《数学与实验统计》

# 第 31 章 · 微分几何与流形优化：切空间、收缩映射、测地线、黎曼梯度

配套讲义：仓库里 `math-and-eval-stats/notes/lessons/31-manifold-optimization.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy / SciPy（Colab 自带），不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一行代码、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 投影 | (I − xxᵀ)w | `proj(x, w)` | [1, 2, 0] |
| 第二步 收缩 vs 指数 | 差的阶 | `retract`、`exp_map` | 3.3e-4 → 3.3e-7 |
| 第三步 测地线 | arccos | `geodesic(x, y)` | 1.5708 vs 1.4142 |
| 第四步 RGD | 投影 + 收缩 | `rgd(x0)` | 16 次，3.644973 |
| 第五步 SO(3) | R·skew(RᵀG) | `so3_tangent(R, G)` | 反对称矩阵 |

例子：单位球面 S² ⊂ R³（最简单的弯曲流形）。在球面上最大化 Rayleigh 商 xᵀAx，答案是 A 的最大特征向量。
里程碑 ME-M22 的“Stiefel / SO(3) 上的黎曼梯度下降”留给自己做；这里只在球面上演示。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import numpy as np
A_MAT = np.array([[3.0, 1.0, 0.0], [1.0, 2.0, 0.5], [0.0, 0.5, 1.0]])

**函数 `proj`**：切空间投影 P_x v = (I − xxᵀ)v：去掉沿法向（x 方向）的分量。

In [ ]:
def proj(x: np.ndarray, v: np.ndarray) -> np.ndarray:
    """切空间投影 P_x v = (I − xxᵀ)v：去掉沿法向（x 方向）的分量。"""
    return v - x * (x @ v)

**函数 `retract`**：收缩映射：先在切平面上走 v，再归一化拉回球面。

In [ ]:
def retract(x: np.ndarray, v: np.ndarray) -> np.ndarray:
    """收缩映射：先在切平面上走 v，再归一化拉回球面。"""
    y = x + v
    return y / np.linalg.norm(y)

**函数 `exp_map`**：指数映射：沿大圆（测地线）走长度 ‖v‖。

In [ ]:
def exp_map(x: np.ndarray, v: np.ndarray) -> np.ndarray:
    """指数映射：沿大圆（测地线）走长度 ‖v‖。"""
    t = np.linalg.norm(v)
    return x if t == 0 else np.cos(t) * x + np.sin(t) * v / t

**函数 `geodesic`**：小工具，把结果整理成好读的样子再打印。

In [ ]:
def geodesic(x: np.ndarray, y: np.ndarray) -> float:
    return float(np.arccos(np.clip(x @ y, -1, 1)))

**函数 `rgd`**：黎曼梯度上升：欧氏梯度 2Ax → 投影到切空间 → 沿它走一步 → 收缩回球面。返回 (x, 迭代次数, 每次的 Rayleigh 商)。

In [ ]:
def rgd(x0: np.ndarray, lr: float = 0.2, iters: int = 200, tol: float = 1e-10) -> tuple[np.ndarray, int, list[float]]:
    """黎曼梯度上升：欧氏梯度 2Ax → 投影到切空间 → 沿它走一步 → 收缩回球面。返回 (x, 迭代次数, 每次的 Rayleigh 商)。"""
    x = x0 / np.linalg.norm(x0)
    vals = [float(x @ A_MAT @ x)]
    for k in range(1, iters + 1):
        g = proj(x, 2 * A_MAT @ x)
        if np.linalg.norm(g) < tol:
            return x, k - 1, vals
        x = retract(x, lr * g)
        vals.append(float(x @ A_MAT @ x))
    return x, iters, vals

**函数 `naive_gd`**：不投影、不收缩，直接在 R³ 里做梯度上升：‖x‖ 会一路变大（xᵀAx 无上界），离开球面。返回最后的 ‖x‖。

In [ ]:
def naive_gd(x0: np.ndarray, lr: float = 0.2, iters: int = 20) -> float:
    """不投影、不收缩，直接在 R³ 里做梯度上升：‖x‖ 会一路变大（xᵀAx 无上界），离开球面。返回最后的 ‖x‖。"""
    x = x0 / np.linalg.norm(x0)
    for _ in range(iters):
        x = x + lr * 2 * A_MAT @ x
    return float(np.linalg.norm(x))

**函数 `so3_tangent`**：SO(3) 上的黎曼梯度（嵌入度量）：把欧氏梯度 G 投影成 R·skew(RᵀG)，skew(M) = (M − Mᵀ)/2。

In [ ]:
def so3_tangent(r: np.ndarray, g: np.ndarray) -> np.ndarray:
    """SO(3) 上的黎曼梯度（嵌入度量）：把欧氏梯度 G 投影成 R·skew(RᵀG)，skew(M) = (M − Mᵀ)/2。"""
    m = r.T @ g
    return r @ ((m - m.T) / 2)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
x = np.array([0.0, 0.0, 1.0])
v = np.array([0.3, 0.0, 0.0])

In [ ]:
print("\n第一步 切空间与投影：球面在 x 处的切平面 = 和 x 垂直的向量")
w = np.array([1.0, 2.0, 3.0])
print(f"  x = [0, 0, 1]，w = [1, 2, 3]：投影 P_x w = {proj(x, w).tolist()}，和 x 的内积 {x @ proj(x, w):.1f}")

In [ ]:
print("\n第二步 收缩映射 vs 指数映射：两种“把切向量拉回流形”的办法，一阶相同")
for t in (0.3, 0.1, 0.01):
    vv = v / 0.3 * t
    d = np.linalg.norm(retract(x, vv) - exp_map(x, vv))
    print(f"  ‖v‖ = {t}：两者相差 {d:.3e}")

In [ ]:
print("\n第三步 测地线距离 vs 弦长")
for ang in (10, 90, 180):
    y = np.array([np.sin(np.radians(ang)), 0, np.cos(np.radians(ang))])
    print(f"  夹角 {ang}°：测地线 {geodesic(x, y):.4f}，弦长 {np.linalg.norm(x - y):.4f}")

In [ ]:
print("\n第四步 黎曼梯度上升：在球面上最大化 xᵀAx")
xs, k, vals = rgd(np.array([1.0, 1.0, 1.0]))
lam, vec = np.linalg.eigh(A_MAT)
print(f"  {k} 次收敛到 x = {np.round(xs, 4).tolist()}，xᵀAx = {vals[-1]:.6f}；最大特征值 {lam[-1]:.6f}")
print(f"  前几次 xᵀAx：{[round(v_, 4) for v_ in vals[:5]]}；每次都在球面上：‖x‖ = {np.linalg.norm(xs):.6f}")
print(f"  不投影不收缩，直接在 R³ 里走 20 步：‖x‖ = {naive_gd(np.array([1.0, 1.0, 1.0])):.3g}")

In [ ]:
print("\n第五步 和第 19 章接上：SO(3) 也是流形，黎曼梯度在 R·so(3) 里")
r = np.eye(3)
g = np.array([[0.0, 1.0, 0.0], [0.0, 0.0, 0.0], [2.0, 0.0, 0.0]])
t = so3_tangent(r, g)
print(f"  R = I，欧氏梯度 G：黎曼梯度 = skew(G) = {t.tolist()}（反对称 ↔ 轴角向量）")

## 3. 代码里才看得到的

1. **停止条件看黎曼梯度的范数**：欧氏梯度 2Ax 在最优点不为 0（它指向法向），只有投影后的部分才会变成 0。
2. **`proj` 写成 v − x(x·v)**：不构造 3×3 的 I − xxᵀ 矩阵，高维时省内存和计算。
3. **`naive_gd` 的 ‖x‖ 爆炸是指数级的**：沿最大特征向量的分量每步乘 1 + 0.4 × 3.64 ≈ 2.46，20 步约 6 × 10⁷ 倍，和打印的 5.5 × 10⁷ 同一量级。

## 4. 自己改一改

1. 把步长改成 1.0 和 0.01，看收敛次数怎么变；步长太大时会不会振荡？
2. 把 A 的最大两个特征值改得很接近，收敛变慢多少？

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 脚下的平地是切空间，投影去掉法向；
> 切空间里走一步，收缩拉回流形，归一化就够；
> 测地线近处像直线，远处差很多；
> 欧氏梯度、投影、走一步、收缩回来；
> 李群也是流形，SO(3) 上是 R 乘 skew。